# Figure S1G — Probabilistic Model Comparison (locked 300-patient test set)

Grouped bar chart of test-set F1 scores (±95% CI) for 4 foundation models across 6 toxicities, with significance annotations vs. Llama 4 Maverick (reference).

**Models:** Llama 4 Maverick, Llama 4 Scout, Claude Sonnet 4.5, GPT-OSS-120B

**Denominator (same as Fig 1B / S1F):** locked `test_mrns.csv` (n=298) + 2 RAG-filtered patients padded as all-negative.

**Data sources:**
- `llama_maverick_test_298_results.csv` — Maverick (locked 298; same file as S1F)
- `combined_298_patients_scout_results.csv` — Scout (valid 281 + new 17)
- `combined_298_patients_claude_results.csv` — Claude (valid 281 + new 17)
- `combined_298_patients_gpt_results.csv` — GPT (valid 281 + new 17)
- `final_gold_standard_1k.csv` — gold standard labels
- `additional_negative_mrns.txt` — 2 padded all-negative patients

`original_1k_test_298_results.csv` is the stale 298 draw and is not used. Thresholds for Scout / Claude / GPT are fit on a 149-patient subset of the locked 298; Maverick uses the Fig 1B thresholds. F1 is reported on the held-out locked-test patients plus the 2 padded patients.

Outputs are written to `figure 1/results/supp/`.


In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

%matplotlib inline

# ---- Hard-fail if Arial isn't actually resolved (no silent fallback) ----
import matplotlib.font_manager as fm
_arial_path = fm.findfont('Arial', fallback_to_default=False)
if 'Arial' not in _arial_path:
    raise RuntimeError(
        f"Arial not found -- matplotlib resolved to '{_arial_path}' instead. "
        "Install Arial or update font.sans-serif before rendering this figure."
    )
print(f"Arial resolved to: {_arial_path}")

In [ ]:
ROOT = Path("..").resolve()
FIGURES = ROOT.parent.parent
DATA = FIGURES / "figures_data" / "figure 1" / "data"
RESULTS = ROOT / "results"
(RESULTS / "supp").mkdir(parents=True, exist_ok=True)

MAV_FILE    = DATA / "llama_maverick_test_298_results.csv"
SCOUT_FILE  = DATA / "combined_298_patients_scout_results.csv"
CLAUDE_FILE = DATA / "combined_298_patients_claude_results.csv"
GPT_FILE    = DATA / "combined_298_patients_gpt_results.csv"
GOLD_FILE   = DATA / "final_gold_standard_1k.csv"
TEST_MRN_FILE = DATA / "test_mrns.csv"
ADDITIONAL_NEG_MRN_FILE = DATA / "additional_negative_mrns.txt"

OUT_PATH = RESULTS / "supp" / "Model_Comparison_S1G.pdf"
CSV_OUT  = RESULTS / "supp" / "Model_Comparison_S1G_results.csv"

all_files = [MAV_FILE, SCOUT_FILE, CLAUDE_FILE, GPT_FILE,
             GOLD_FILE, TEST_MRN_FILE, ADDITIONAL_NEG_MRN_FILE]
for p in all_files:
    assert p.exists(), f"Missing: {p}"
print("All input files found.")
print(f"Data: {DATA}")
print(f"Maverick: {MAV_FILE.name}")
print(f"Scout:    {SCOUT_FILE.name}")
print(f"Claude:   {CLAUDE_FILE.name}")
print(f"GPT:      {GPT_FILE.name}")


In [ ]:
# ---------------------------------------------------------------------------
# CONSTANTS
# ---------------------------------------------------------------------------
TOXICITIES = [
    "liver_toxicity",
    "hypothyroidism",
    "pneumonitis",
    "colitis",
    "adrenal_insufficiency",
    "hyperthyroidism",
]

DISPLAY_NAMES = {
    "liver_toxicity":        "Liver\nToxicity",
    "hypothyroidism":        "Hypo-\nthyroidism",
    "pneumonitis":           "Pneumonitis",
    "colitis":               "Colitis",
    "adrenal_insufficiency": "Adrenal\nInsufficiency",
    "hyperthyroidism":       "Hyper-\nthyroidism",
}

MODEL_ORDER = ["Llama 4 Maverick", "Llama 4 Scout", "Claude Sonnet 4.5", "GPT-OSS-120B"]
MODEL_COLORS = {
    "Llama 4 Maverick":  "#4878CF",
    "Llama 4 Scout":     "#D65F5F",
    "Claude Sonnet 4.5": "#6ACC64",
    "GPT-OSS-120B":      "#B47CC7",
}

MAV_THRESH = {
    "pneumonitis":           0.710,
    "adrenal_insufficiency": 0.810,
    "liver_toxicity":        0.010,
    "colitis":               0.710,
    "hyperthyroidism":       0.810,
    "hypothyroidism":        0.510,
}

TRAIN_SIZE = 149
SPLIT_SEED = 123
N_BOOT     = 2000
SEED_BASE  = 1000

In [ ]:
COL_RENAME = {
    "liver toxicity": "liver_toxicity",
    "adrenal insufficiency": "adrenal_insufficiency",
}

def _norm_mrn(s):
    return s.astype(str).str.replace(r"\.0$", "", regex=True).str.strip().str.zfill(8)

def _read_csv(path):
    return pd.read_csv(path, low_memory=False)

def collapse_model(path):
    """Load one long CSV and collapse to patient-level max per toxicity."""
    df = _read_csv(path).rename(columns=COL_RENAME)
    df["mrn"] = _norm_mrn(df["mrn"])
    missing = [t for t in TOXICITIES if t not in df.columns]
    if missing:
        raise KeyError(f"{path.name} missing columns: {missing}")
    return df.groupby("mrn")[TOXICITIES].max().sort_index()

def load_gold():
    g = _read_csv(GOLD_FILE).rename(columns=COL_RENAME)
    mrn_col = next((c for c in ["MRN", "mrn", "MRN_STR", "mrn_str"] if c in g.columns), None)
    if mrn_col is None:
        raise ValueError("MRN column not found in gold standard CSV.")
    g["mrn"] = _norm_mrn(g[mrn_col])
    g = g.drop_duplicates("mrn").set_index("mrn")
    for t in TOXICITIES:
        if t in g.columns:
            g[t] = pd.to_numeric(g[t], errors="coerce").fillna(0).round().astype(int)
        else:
            g[t] = 0
    return g[TOXICITIES]


In [ ]:
def f1_at_threshold(y_true, y_score, thr):
    y_pred = (y_score >= (thr - 1e-10)).astype(int)
    return float(f1_score(y_true, y_pred, zero_division=0))

def find_optimal_threshold(y_true, y_score):
    grid = np.linspace(0.0, 1.0, 101)
    best_thr, best_f1 = 0.5, -1.0
    for thr in grid:
        f = f1_at_threshold(y_true, y_score, float(thr))
        if f > best_f1 + 1e-12 or (abs(f - best_f1) <= 1e-12 and thr < best_thr):
            best_f1 = f
            best_thr = float(thr)
    return best_thr

def format_pval(p):
    if p < 0.001:
        return "P < 0.001"
    elif p < 0.01:
        return f"P = {p:.3f}"
    elif p < 0.05:
        return f"P = {p:.2f}"
    else:
        return f"P = {p:.2f}"

def significance_stars(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"


In [ ]:
# ---------------------------------------------------------------------------
# Load locked 298 (same patients as Fig 1B / S1F), then pad 2 all-negatives
# ---------------------------------------------------------------------------
gold   = load_gold()
mav    = collapse_model(MAV_FILE)
scout  = collapse_model(SCOUT_FILE)
claude = collapse_model(CLAUDE_FILE)
gpt    = collapse_model(GPT_FILE)

test_df = _read_csv(TEST_MRN_FILE)
test_col = next(c for c in test_df.columns if c.lower() in {"mrn", "mrn_str"})
test_ids = list(dict.fromkeys(_norm_mrn(test_df[test_col]).tolist()))
assert len(test_ids) == 298, f"Expected 298 locked-test patients, got {len(test_ids)}"

for name, scores in [("Maverick", mav), ("Scout", scout), ("Claude", claude), ("GPT", gpt), ("gold", gold)]:
    missing = set(test_ids) - set(scores.index)
    assert not missing, f"{name} missing {len(missing)} locked-test patients"

gold   = gold.loc[test_ids]
mav    = mav.loc[test_ids]
scout  = scout.loc[test_ids]
claude = claude.loc[test_ids]
gpt    = gpt.loc[test_ids]

ADDITIONAL_NEG_MRNS = [
    line.strip().zfill(8)
    for line in ADDITIONAL_NEG_MRN_FILE.read_text().splitlines() if line.strip()
]
assert len(ADDITIONAL_NEG_MRNS) == 2, f"Expected 2 padding patients, got {len(ADDITIONAL_NEG_MRNS)}"
assert not (set(ADDITIONAL_NEG_MRNS) & set(test_ids)), "Padding patient already in locked split"

n_added = 0
for mrn in ADDITIONAL_NEG_MRNS:
    if mrn in mav.index:
        continue
    zero_score = pd.DataFrame([[0.0] * len(TOXICITIES)], columns=TOXICITIES, index=[mrn])
    zero_truth = zero_score.astype(int)
    mav    = pd.concat([mav, zero_score])
    scout  = pd.concat([scout, zero_score])
    claude = pd.concat([claude, zero_score])
    gpt    = pd.concat([gpt, zero_score])
    gold   = pd.concat([gold, zero_truth])
    n_added += 1

assert len(mav) == 300, f"Expected 300, got {len(mav)}"
print(f"Locked test: {len(test_ids)}")
print(f"Padded RAG-filtered all-negative patients: {n_added}")
print(f"Test-set denominator after padding: {len(mav)}")

# Thresholds for Scout / Claude / GPT are fit on a subset of the locked 298.
# Padded patients are evaluation-only (all-negative, never an LLM call).
rng_split = np.random.default_rng(SPLIT_SEED)
idx = np.arange(len(test_ids))
rng_split.shuffle(idx)
train_idx = idx[:TRAIN_SIZE]
test_idx  = idx[TRAIN_SIZE:]
mrn_train = [test_ids[i] for i in train_idx]
mrn_test  = [test_ids[i] for i in test_idx] + ADDITIONAL_NEG_MRNS

print(f"Threshold-tuning subset: {len(mrn_train)}")
print(f"F1 evaluation set (held-out locked test + pad): {len(mrn_test)}")


In [ ]:
# ---------------------------------------------------------------------------
# Compute metrics + paired bootstrap p-values (each model vs Maverick)
# ---------------------------------------------------------------------------
gold_tr = gold.loc[mrn_train]
gold_te = gold.loc[mrn_test]

model_data = {
    "Llama 4 Maverick": {"train": mav.loc[mrn_train],    "test": mav.loc[mrn_test]},
    "Llama 4 Scout":    {"train": scout.loc[mrn_train],   "test": scout.loc[mrn_test]},
    "Claude Sonnet 4.5": {"train": claude.loc[mrn_train], "test": claude.loc[mrn_test]},
    "GPT-OSS-120B":     {"train": gpt.loc[mrn_train],     "test": gpt.loc[mrn_test]},
}

# Step 1: Find thresholds
thresholds = {}
for model_name, data in model_data.items():
    thresholds[model_name] = {}
    for tox in TOXICITIES:
        y_tr = gold_tr[tox].to_numpy(int)
        s_tr = data["train"][tox].to_numpy(float)
        if model_name == "Llama 4 Maverick":
            thr = MAV_THRESH[tox]
        else:
            if y_tr.sum() == 0:
                thr = 0.5
            else:
                thr = find_optimal_threshold(y_tr, s_tr)
        thresholds[model_name][tox] = thr

# Step 2: Paired bootstrap — Maverick vs each other model
# For each toxicity, resample test set patients (stratified), compute F1 for
# both Maverick and the comparison model using the SAME indices.

results = {m: [] for m in MODEL_ORDER}  # list of dicts per toxicity
p_values = {}  # p_values[(model, tox)] = p

for tox in TOXICITIES:
    y_te = gold_te[tox].to_numpy(int)
    pos_idx = np.where(y_te == 1)[0]
    neg_idx = np.where(y_te == 0)[0]

    # Generate shared bootstrap indices once per toxicity
    rng = np.random.default_rng(SEED_BASE)
    boot_indices = []
    for b in range(N_BOOT):
        s_pos = rng.choice(pos_idx, size=len(pos_idx), replace=True) if len(pos_idx) > 0 else pos_idx
        s_neg = rng.choice(neg_idx, size=len(neg_idx), replace=True)
        boot_indices.append(np.concatenate([s_pos, s_neg]))

    # Compute bootstrap F1 for every model
    boot_f1_all = {}
    for model_name in MODEL_ORDER:
        s_te = model_data[model_name]["test"][tox].to_numpy(float)
        thr = thresholds[model_name][tox]
        point_f1 = f1_at_threshold(y_te, s_te, thr)

        boot_f1 = np.empty(N_BOOT)
        for b in range(N_BOOT):
            idx = boot_indices[b]
            boot_f1[b] = f1_at_threshold(y_te[idx], s_te[idx], thr)

        lo = float(np.percentile(boot_f1, 2.5))
        hi = float(np.percentile(boot_f1, 97.5))
        boot_f1_all[model_name] = boot_f1

        results[model_name].append({
            "tox": tox, "F1": point_f1, "lo": lo, "hi": hi, "thr": thr
        })

    # Paired p-values: Maverick vs each other model
    mav_boot = boot_f1_all["Llama 4 Maverick"]
    for model_name in MODEL_ORDER[1:]:  # skip Maverick
        delta = mav_boot - boot_f1_all[model_name]
        p_val = 2 * min(np.mean(delta >= 0), np.mean(delta <= 0))
        p_val = min(p_val, 1.0)
        p_values[(model_name, tox)] = p_val

# Print summary
for tox in TOXICITIES:
    print(f"\n{tox}:")
    for model_name in MODEL_ORDER:
        r = results[model_name]
        entry = [e for e in r if e["tox"] == tox][0]
        if model_name == "Llama 4 Maverick":
            print(f"  {model_name:<22} F1={entry['F1']:.3f} [{entry['lo']:.3f}-{entry['hi']:.3f}] (reference)")
        else:
            p = p_values[(model_name, tox)]
            print(f"  {model_name:<22} F1={entry['F1']:.3f} [{entry['lo']:.3f}-{entry['hi']:.3f}]  "
                  f"vs Mav: {format_pval(p)} {significance_stars(p)}")

rows = []
for model_name in MODEL_ORDER:
    for entry in results[model_name]:
        rec = {
            "model": model_name,
            "toxicity": entry["tox"],
            "toxicity_display": DISPLAY_NAMES[entry["tox"]].replace("\n", " "),
            "threshold": entry["thr"],
            "f1": entry["F1"],
            "f1_ci_lo": entry["lo"],
            "f1_ci_hi": entry["hi"],
            "n_eval": len(mrn_test),
        }
        if model_name == "Llama 4 Maverick":
            rec["p_vs_maverick"] = None
            rec["p_vs_maverick_label"] = "reference"
        else:
            rec["p_vs_maverick"] = p_values[(model_name, entry["tox"])]
            rec["p_vs_maverick_label"] = format_pval(p_values[(model_name, entry["tox"])])
        rows.append(rec)
results_df = pd.DataFrame(rows)
results_df.to_csv(CSV_OUT, index=False)
print(f"\nSaved: {CSV_OUT.name}")


In [ ]:
# ---------------------------------------------------------------------------
# Figure with significance asterisks
# ---------------------------------------------------------------------------
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 6,
    "axes.labelsize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "legend.fontsize": 5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

x = np.arange(len(TOXICITIES))
bar_width = 0.18

fig, ax = plt.subplots(figsize=(3.9, 2.3))  

for i, model in enumerate(MODEL_ORDER):
    f1_vals = np.array([e["F1"] for e in results[model]])
    lo_vals = np.array([e["lo"] for e in results[model]])
    hi_vals = np.array([e["hi"] for e in results[model]])
    yerr = np.vstack([np.maximum(0.0, f1_vals - lo_vals),
                      np.maximum(0.0, hi_vals - f1_vals)])

    bar_positions = x + (i - 1.5) * bar_width

    ax.bar(bar_positions, f1_vals, bar_width,
           yerr=yerr, capsize=2,
           color=MODEL_COLORS.get(model, "#333333"),
           edgecolor="white", linewidth=0.4, label=model, alpha=0.9,
           error_kw={"linewidth": 0.6, "capthick": 0.6})

    # Add significance asterisks — only when significant
    if model != "Llama 4 Maverick":
        for j, tox in enumerate(TOXICITIES):
            p = p_values[(model, tox)]
            stars = significance_stars(p)
            if stars != "ns":
                bar_top = f1_vals[j] + yerr[1, j]
                ax.text(bar_positions[j], bar_top + 0.02, stars,
                        ha="center", va="bottom", fontsize=5, fontfamily="Arial",
                        fontweight="bold")

ax.set_ylabel("F1 Score (test set)")
ax.set_xticks(x)
ax.set_xticklabels([DISPLAY_NAMES[t] for t in TOXICITIES], ha="center")
ax.set_ylim(0, 1.20)
ax.set_yticks(np.arange(0, 1.1, 0.2))

for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
for spine in ("bottom", "left"):
    ax.spines[spine].set_linewidth(0.6)
ax.tick_params(width=0.6, length=3)

# Legend below plot
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15),
          ncol=len(MODEL_ORDER), frameon=True, framealpha=1.0, edgecolor="#cccccc")

plt.tight_layout()
fig.savefig(OUT_PATH, format="pdf", dpi=450)
print(f"Saved: {OUT_PATH.name}")
plt.show()